In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print("Train rows and columns:", train.shape)
print("Test rows and columns:", test.shape)
train.head()

Train rows and columns: (3207, 14)
Test rows and columns: (802, 13)


,id,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,price,engine_displacement_l,engine_cylinders,transmission_type
0,1,Lexus,IS 300 Base,2018,"50,992 mi.",Gasoline,White,Gray,At least 1 accident or damage reported,Yes,"$28,000",3.5,6.0,Automatic
1,2,Chevrolet,Impala Base,2004,"64,500 mi.",Gasoline,Beige,Beige,None reported,Yes,"$5,900",3.4,6.0,Automatic
2,3,RAM,2500 SLT,2017,"86,000 mi.",Diesel,Gray,Gray,At least 1 accident or damage reported,Yes,"$41,000",6.7,6.0,Automatic
3,4,Mercedes-Benz,SL-Class SL 550,2013,"24,933 mi.",Gasoline,Silver,Red,At least 1 accident or damage reported,Yes,"$40,250",4.6,8.0,Other
4,5,Ford,Shelby GT350R Base,2018,"18,500 mi.",Gasoline,Blue,Black,At least 1 accident or damage reported,Yes,"$77,999",5.2,8.0,Manual


In [3]:
train["price"] = train["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
train["mileage"] = train["milage"].str.replace(" mi.", "", regex=False).str.replace(",", "", regex=False).astype(float)
test["mileage"] = test["milage"].str.replace(" mi.", "", regex=False).str.replace(",", "", regex=False).astype(float)

In [4]:
print(train.isnull().sum())

id                         0
brand                      0
model                      0
model_year                 0
milage                     0
fuel_type                133
ext_col                    0
int_col                    0
accident                  92
clean_title              482
price                      0
engine_displacement_l    175
engine_cylinders         351
transmission_type          0
mileage                    0
dtype: int64


In [5]:
train = train[train["price"] <= 300000]
print("Rows left after removing very expensive cars:", len(train))

Rows left after removing very expensive cars: 3183


In [6]:
engine_size_median = train["engine_displacement_l"].median()
cylinders_median = train["engine_cylinders"].median()

for data in [train, test]:
    data["engine_displacement_l"] = data["engine_displacement_l"].fillna(engine_size_median)
    data["engine_cylinders"] = data["engine_cylinders"].fillna(cylinders_median)
    data["fuel_type"] = data["fuel_type"].fillna("Unknown")
    data["accident"] = data["accident"].fillna("None reported")
    data["accident"] = data["accident"].str.startswith("At least").astype(int)
    data["clean_title"] = data["clean_title"].notna().astype(int)
    data["age"] = 2025 - data["model_year"]

print(train.isnull().sum().sum(), "missing values left in train")

0 missing values left in train


In [7]:
columns = ["age", "mileage", "engine_displacement_l", "engine_cylinders",
           "accident", "clean_title", "brand", "fuel_type", "transmission_type"]

X = train[columns]
y = train["price"]
X_new = test[columns]

X = pd.get_dummies(X, columns=["brand", "fuel_type", "transmission_type"], dtype=int)
X_new = pd.get_dummies(X_new, columns=["brand", "fuel_type", "transmission_type"], dtype=int)
X_new = X_new.reindex(columns=X.columns, fill_value=0)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training cars:", len(X_train))
print("Testing cars:", len(X_test))

Training cars: 2546
Testing cars: 637


In [9]:
linear = LinearRegression()
linear.fit(X_train, y_train)
r2_linear = r2_score(y_test, linear.predict(X_test))
print("Linear Regression R2:", round(r2_linear, 4))

Linear Regression R2: 0.5832


In [10]:
tree = DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=42)
tree.fit(X_train, y_train)
r2_tree = r2_score(y_test, tree.predict(X_test))
print("Decision Tree R2:", round(r2_tree, 4))

Decision Tree R2: 0.6961


In [11]:
forest = RandomForestRegressor(n_estimators=300, random_state=42)
forest.fit(X_train, y_train)
r2_forest = r2_score(y_test, forest.predict(X_test))
print("Random Forest R2:", round(r2_forest, 4))

Random Forest R2: 0.8024


In [12]:
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
r2_knn = r2_score(y_test, knn.predict(X_test_scaled))
print("KNN R2:", round(r2_knn, 4))

KNN R2: 0.58


In [13]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest", "KNN"],
    "R2 Score": [r2_linear, r2_tree, r2_forest, r2_knn]
})
results = results.sort_values("R2 Score", ascending=False).reset_index(drop=True)
print(results.round(4))
print()
print("Best model:", results.loc[0, "Model"])
print("Best R2 score:", round(results.loc[0, "R2 Score"], 4))

               Model  R2 Score
0      Random Forest    0.8024
1      Decision Tree    0.6961
2  Linear Regression    0.5832
3                KNN    0.5800

Best model: Random Forest
Best R2 score: 0.8024


In [14]:
best_model = RandomForestRegressor(n_estimators=300, random_state=42)
best_model.fit(X, y)
predicted_price = best_model.predict(X_new)

submission = pd.DataFrame({"id": test["id"], "price": predicted_price.round(2)})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv with", len(submission), "rows")
submission.head()

Saved submission.csv with 802 rows


,id,price
0,3208,32971.47
1,3209,30424.85
2,3210,25244.45
3,3211,41649.97
4,3212,19578.26
